<a href="https://www.kaggle.com/code/insanejsk/reranker-data-prep?scriptVersionId=327050476" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

### Reranker Dataset creation

For each MS Marco Passage
  - 1 positive passage  (is_selected=1)
  - up to 9 negative passages (is_selected=0)
  - generates up to 9 (query, positive, negative) pairs

Output:
  - reranker_train.jsonl   (~1.4-1.6M pairs from 200k training split)
  - reranker_val.jsonl     (5k queries × up to 9 pairs = ~45k pairs)

Kaggle output path: /kaggle/working/

In [ ]:
import json
import os
from datasets import load_dataset

OUT_DIR = "/kaggle/working/"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
print("Loading MS MARCO passage ranking")
dataset = load_dataset("microsoft/ms_marco", "v2.1")
train_data = dataset["train"]
val_data   = dataset["validation"]

print(f"Train samples : {len(train_data):,}")
print(f"Val samples   : {len(val_data):,}")

In [ ]:
# Exhaustive pairs
def build_pairs(data, out_path, max_queries=None, desc=""):
    """
    For each query:
      - find the positive passage (is_selected=1)
      - pair it with every negative passage (is_selected=0)
      - write one JSONL line per pair
    Skips queries with no selected passage.
    """
    total_queries = 0
    total_pairs   = 0
    skipped       = 0

    with open(out_path, "w") as f:
        for i, sample in enumerate(data):
            if max_queries and total_queries >= max_queries:
                break

            query    = sample["query"]
            passages = sample["passages"]

            is_selected  = passages["is_selected"]    # list of 0/1
            passage_text = passages["passage_text"]   # parallel list of strings

            # find positive
            positives = [
                passage_text[j]
                for j, sel in enumerate(is_selected)
                if sel == 1
            ]

            if not positives:
                skipped += 1
                continue
            positive = positives[0]

            # find all negatives
            negatives = [
                passage_text[j]
                for j, sel in enumerate(is_selected)
                if sel == 0
            ]
            if not negatives:
                skipped += 1
                continue

            # write one pair per negative
            for neg in negatives:
                f.write(json.dumps({
                    "query"    : query,
                    "positive" : positive,
                    "negative" : neg,
                }) + "\n")
                total_pairs += 1

            total_queries += 1

            if (total_queries) % 50_000 == 0:
                print(f"{desc} | Queries: {total_queries:,} | Pairs: {total_pairs:,}")

    print(f"\n{desc} complete")
    print(f"Queries processed: {total_queries:,}")
    print(f"Queries skipped  : {skipped:,}  (no pos passage)")
    print(f"Total pairs      : {total_pairs:,}")
    print(f"Avg pairs/query  : {total_pairs/max(total_queries,1):.1f}")
    print(f"Saved to         : {out_path}\n")

    return total_pairs

In [ ]:
# Train set
print("Reranker training pairs...")
train_path  = os.path.join(OUT_DIR, "reranker_train.jsonl")
train_pairs = build_pairs(
    train_data,
    train_path,
    max_queries=200000,   # use 200k queries from training set
    desc="Train",
)

In [ ]:
# Val set
print("Reranker validation pairs...")
val_path  = os.path.join(OUT_DIR, "reranker_val.jsonl")
val_pairs = build_pairs(
    val_data,
    val_path,
    max_queries=5000,  # 5k queries for eval
    desc="Val",
)

In [ ]:
# Sanity test
print("First 3 training pairs:")
with open(train_path) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        obj = json.loads(line)
        print(f"Pair {i+1}")
        print(f"Query   : {obj['query'][:80]}")
        print(f"Positive: {obj['positive'][:80]}")
        print(f"Negative: {obj['negative'][:80]}")

In [ ]:
print(f"\n{'='*40}")
print(f"Dataset Summary")
print(f"Train pairs: {train_pairs:,}")
print(f"Val pairs  : {val_pairs:,}")
print(f"Train file : {train_path}")
print(f"Val file   : {val_path}")
print(f"{'='*40}")